In [1]:
from pathlib import Path
from collections import Counter

import ifcopenshell
import ifcopenshell.util.element
import ifcopenshell.util.unit
import pandas as pd

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)

IFC_PATH = ROOT / "data/raw/viadukten/viadukten.ifc"

assert IFC_PATH.is_file(), f"File not found: {IFC_PATH}"

model = ifcopenshell.open(str(IFC_PATH))

print(f"File: {IFC_PATH.name}")
print(f"Schema: {model.schema}")
print(f"File size: {IFC_PATH.stat().st_size / 1024:.2f} KB")

File: viadukten.ifc
Schema: IFC4
File size: 207.49 KB


In [3]:
project = model.by_type("IfcProject")[0]

print(f"Project name: {project.Name}")
print(f"Project ID: {project.GlobalId}")
print(f"Description: {project.Description}")

Project name: SLJ-GRH-D1
SLJ-GRH-D1
SLJ-GRH-D1
Project ID: 11nXSKglbF0Raz7B$ue1yA
Description: None


In [4]:
elements = model.by_type("IfcElement")

entity_counts = Counter(
    element.is_a() for element in elements
)

df_entities = pd.DataFrame(
    entity_counts.items(),
    columns=["IFC Class", "Count"]
).sort_values("Count", ascending=False)

display(df_entities)

print(f"Total elements: {len(elements)}")


,IFC Class,Count
3,IfcSlab,13
2,IfcColumn,12
0,IfcBeam,6
4,IfcWall,2
1,IfcBuildingElementProxy,1


Total elements: 34


In [5]:
proxies = model.by_type("IfcBuildingElementProxy")

for element in proxies:
    print(f"Name: {element.Name}")
    print(f"GlobalId: {element.GlobalId}")
    print(f"Description: {element.Description}")
    print(f"Object Type: {element.ObjectType}")
    print(f"Predefined Type: {element.PredefinedType}")
    print(f"Tag: {element.Tag}")

Name: Model Lines:1681012
GlobalId: 3h0RsZPknDw9WCR$a1yEUL
Description: None
Object Type: None
Predefined Type: None
Tag: 1681012


In [6]:
records = []

for element in elements:
    records.append({
        "global_id": element.GlobalId,
        "ifc_class": element.is_a(),
        "name": element.Name,
        "description": element.Description,
        "object_type": element.ObjectType,
        "predefined_type": getattr(
            element, "PredefinedType", None
        ),
        "tag": getattr(element, "Tag", None),
    })

df_elements = pd.DataFrame(records)

display(df_elements)

,global_id,ifc_class,name,description,object_type,predefined_type,tag
0,261wa1zhHD7uN1f7Gb7exb,IfcBeam,Concrete-Rectangular Beam:STI_Balk 1600x700:16...,None,None,None,1679022
1,3YQplDvTD6mBz6b1Bw99Fa,IfcBeam,Concrete-Rectangular Beam:STI_Balk 1600x700:16...,None,None,None,1679519
2,3YQplDvTD6mBz6b1Bw999y,IfcBeam,Concrete-Rectangular Beam:STI_Balk 1600x700:16...,None,None,None,1679623
3,3YQplDvTD6mBz6b1Bw999n,IfcBeam,Concrete-Rectangular Beam:STI_Balk 1600x700:16...,None,None,None,1679626
4,3YQplDvTD6mBz6b1Bw999s,IfcBeam,Concrete-Rectangular Beam:STI_Balk 1600x700:16...,None,None,None,1679629
5,3YQplDvTD6mBz6b1Bw999h,IfcBeam,Concrete-Rectangular Beam:STI_Balk 1600x700:16...,None,None,None,1679632
6,3h0RsZPknDw9WCR$a1yEUL,IfcBuildingElementProxy,Model Lines:1681012,None,None,None,1681012
7,2MkfNJG9b9mQhC0P8C4Dpd,IfcColumn,M_Concrete-Rectangular-Column:STI_Betong 1200x...,None,None,None,1677653
8,2MkfNJG9b9mQhC0P8C4Dpa,IfcColumn,M_Concrete-Rectangular-Column:STI_Betong 1200x...,None,None,None,1677654
9,2MkfNJG9b9mQhC0P8C4Dpg,IfcColumn,M_Concrete-Rectangular-Column:STI_Betong 1200x...,None,None,None,1677656


In [7]:
def classify_element(element):
    ifc_class = element.is_a()
    name = (element.Name or "").lower()

    if ifc_class == "IfcBuildingElementProxy":
        if "model lines" in name:
            return "Reference Geometry"

    if ifc_class == "IfcColumn":
        return "Columns"

    if ifc_class == "IfcBeam":
        return "Beams"

    if ifc_class == "IfcFooting":
        return "Foundations"

    if ifc_class == "IfcSlab":
        if "footing" in name or "plint" in name:
            return "Foundations"

        if "sti_bro" in name:
            return "Bridge Deck"

        return "Slabs"

    if ifc_class == "IfcWall":
        return "Walls"

    return "Unclassified"


df_elements["category"] = [
    classify_element(element)
    for element in elements
]

display(df_elements["category"].value_counts())


category
Columns               12
Foundations           12
Beams                  6
Walls                  2
Reference Geometry     1
Bridge Deck            1
Name: count, dtype: int64

In [8]:
from ifcopenshell.util.element import get_materials

def extract_materials(element):
    materials = get_materials(element)

    names = [
        material.Name
        for material in materials
        if material.Name
    ]

    return ", ".join(dict.fromkeys(names)) or None


df_elements["material"] = [
    extract_materials(element)
    for element in elements
]

display(
    df_elements.groupby(
        ["category", "material"],
        dropna=False
    ).size().reset_index(name="count")
)

,category,material,count
0,Beams,"Concrete, Cast-in-Place gray",6
1,Bridge Deck,"Concrete, Cast-in-Place gray",1
2,Columns,"Concrete, Cast-in-Place gray",12
3,Foundations,"Concrete, Cast-in-Place gray",12
4,Reference Geometry,NaN,1
5,Walls,"Concrete, Cast-in-Place gray",2


In [9]:
structural_categories = {
    "Columns",
    "Beams",
    "Foundations",
    "Bridge Deck",
    "Walls",
}

df_structural = df_elements[
    df_elements["category"].isin(structural_categories)
].copy()

print("Total IFC elements:", len(df_elements))
print("Structural elements:", len(df_structural))

print(
    "Duplicate GlobalIds:",
    df_elements["global_id"].duplicated().sum()
)

print(
    "Missing GlobalIds:",
    df_elements["global_id"].isna().sum()
)

print(
    "Missing structural materials:",
    df_structural["material"].isna().sum()
)

Total IFC elements: 34
Structural elements: 33
Duplicate GlobalIds: 0
Missing GlobalIds: 0
Missing structural materials: 0


In [10]:
from ifcopenshell.util.unit import calculate_unit_scale

length_scale = calculate_unit_scale(model, "LENGTHUNIT")
area_scale = calculate_unit_scale(model, "AREAUNIT")
volume_scale = calculate_unit_scale(model, "VOLUMEUNIT")

print("Length scale:", length_scale)
print("Area scale:", area_scale)
print("Volume scale:", volume_scale)

Length scale: 0.001
Area scale: 1.0
Volume scale: 1.0


In [11]:
from ifcopenshell.util.element import get_psets
from pprint import pprint

for category in sorted(structural_categories):

    element = next(
        e for e in elements
        if classify_element(e) == category
    )

    print(f"\n--- {category} ---")

    quantities = get_psets(
        element,
        qtos_only=True
    )

    pprint(quantities)


--- Beams ---
{'Qto_BeamBaseQuantities': {'CrossSectionArea': 1.1199999999999992,
                            'GrossSurfaceArea': 59.28,
                            'GrossVolume': 13.887999999999993,
                            'Length': 12400.0,
                            'NetSurfaceArea': 59.280000000000086,
                            'NetVolume': 13.888000000000012,
                            'OuterSurfaceArea': 57.04,
                            'id': 1456}}

--- Bridge Deck ---
{'Qto_SlabBaseQuantities': {'Depth': 1500.0,
                            'GrossArea': 2430.000000000004,
                            'GrossVolume': 3645.0000000000064,
                            'Length': 16200.00000000003,
                            'NetArea': 2430.000000000004,
                            'NetVolume': 3645.000000000007,
                            'Perimeter': 332400.0000000001,
                            'Width': 150000.0,
                            'id': 1861}}

--- Columns ---


In [12]:
explicit_units = [
    quantity
    for quantity in model.by_type("IfcPhysicalSimpleQuantity")
    if quantity.Unit is not None
]

print("Quantities with explicit units:", len(explicit_units))

for quantity in explicit_units[:10]:
    print(quantity.Name, quantity.Unit)

Quantities with explicit units: 0


In [13]:
def extract_quantities(element):
    quantity_sets = get_psets(
        element,
        qtos_only=True
    )

    result = {}

    for set_name, values in quantity_sets.items():
        if not set_name.startswith("Qto_"):
            continue

        for key, value in values.items():
            if key == "id":
                continue

            if isinstance(value, (int, float)):
                result[key] = value

    return result


quantity_records = []

for element in elements:
    quantities = extract_quantities(element)

    quantity_records.append({
        "global_id": element.GlobalId,
        **quantities
    })

df_quantities = pd.DataFrame(quantity_records)

display(df_quantities)

,global_id,Length,CrossSectionArea,OuterSurfaceArea,GrossSurfaceArea,NetSurfaceArea,GrossVolume,NetVolume,Width,Depth,Perimeter,GrossArea,NetArea,Height,GrossFootprintArea,GrossSideArea,NetSideArea
0,261wa1zhHD7uN1f7Gb7exb,12400.0,1.12,57.04,59.28,59.28,13.888,13.888,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3YQplDvTD6mBz6b1Bw99Fa,12400.0,1.12,57.04,59.28,59.28,13.888,13.888,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3YQplDvTD6mBz6b1Bw999y,12400.0,1.12,57.04,59.28,59.28,13.888,13.888,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3YQplDvTD6mBz6b1Bw999n,12400.0,1.12,57.04,59.28,59.28,13.888,13.888,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3YQplDvTD6mBz6b1Bw999s,12400.0,1.12,57.04,59.28,59.28,13.888,13.888,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,3YQplDvTD6mBz6b1Bw999h,12400.0,1.12,57.04,59.28,59.28,13.888,13.888,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,3h0RsZPknDw9WCR$a1yEUL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2MkfNJG9b9mQhC0P8C4Dpd,5200.0,1.44,24.96,NaN,NaN,7.488,7.488,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2MkfNJG9b9mQhC0P8C4Dpa,5200.0,1.44,24.96,NaN,NaN,7.488,7.488,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2MkfNJG9b9mQhC0P8C4Dpg,5200.0,1.44,24.96,NaN,NaN,7.488,7.488,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
quantity_mapping = {
    "Length": ("length_m", length_scale),
    "Width": ("width_m", length_scale),
    "Height": ("height_m", length_scale),
    "Depth": ("depth_m", length_scale),
    "NetVolume": ("net_volume_m3", volume_scale),
    "GrossVolume": ("gross_volume_m3", volume_scale),
    "NetArea": ("net_area_m2", area_scale),
    "NetSideArea": ("net_side_area_m2", area_scale),
    "NetSurfaceArea": ("net_surface_area_m2", area_scale),
    "CrossSectionArea": ("cross_section_area_m2", area_scale),
}

df_normalized = df_quantities[["global_id"]].copy()

for source, (target, scale) in quantity_mapping.items():
    if source in df_quantities.columns:
        df_normalized[target] = df_quantities[source] * scale
    else:
        df_normalized[target] = float("nan")

display(df_normalized.round(3))

,global_id,length_m,width_m,height_m,depth_m,net_volume_m3,gross_volume_m3,net_area_m2,net_side_area_m2,net_surface_area_m2,cross_section_area_m2
0,261wa1zhHD7uN1f7Gb7exb,12.4,NaN,NaN,NaN,13.888,13.888,NaN,NaN,59.28,1.12
1,3YQplDvTD6mBz6b1Bw99Fa,12.4,NaN,NaN,NaN,13.888,13.888,NaN,NaN,59.28,1.12
2,3YQplDvTD6mBz6b1Bw999y,12.4,NaN,NaN,NaN,13.888,13.888,NaN,NaN,59.28,1.12
3,3YQplDvTD6mBz6b1Bw999n,12.4,NaN,NaN,NaN,13.888,13.888,NaN,NaN,59.28,1.12
4,3YQplDvTD6mBz6b1Bw999s,12.4,NaN,NaN,NaN,13.888,13.888,NaN,NaN,59.28,1.12
5,3YQplDvTD6mBz6b1Bw999h,12.4,NaN,NaN,NaN,13.888,13.888,NaN,NaN,59.28,1.12
6,3h0RsZPknDw9WCR$a1yEUL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2MkfNJG9b9mQhC0P8C4Dpd,5.2,NaN,NaN,NaN,7.488,7.488,NaN,NaN,NaN,1.44
8,2MkfNJG9b9mQhC0P8C4Dpa,5.2,NaN,NaN,NaN,7.488,7.488,NaN,NaN,NaN,1.44
9,2MkfNJG9b9mQhC0P8C4Dpg,5.2,NaN,NaN,NaN,7.488,7.488,NaN,NaN,NaN,1.44


In [15]:
df_bim = df_elements.merge(
    df_normalized,
    on="global_id",
    how="left",
    validate="one_to_one"
)

df_bim["is_structural"] = (
    df_bim["category"].isin(structural_categories)
)

df_bim_structural = df_bim[
    df_bim["is_structural"]
].copy()

display(df_bim_structural.round(3))

print("Dataset shape:", df_bim.shape)
print("Structural elements:", len(df_bim_structural))

,global_id,ifc_class,name,description,object_type,predefined_type,tag,category,material,length_m,width_m,height_m,depth_m,net_volume_m3,gross_volume_m3,net_area_m2,net_side_area_m2,net_surface_area_m2,cross_section_area_m2,is_structural
0,261wa1zhHD7uN1f7Gb7exb,IfcBeam,Concrete-Rectangular Beam:STI_Balk 1600x700:16...,None,None,None,1679022,Beams,"Concrete, Cast-in-Place gray",12.4,NaN,NaN,NaN,13.888,13.888,NaN,NaN,59.28,1.12,True
1,3YQplDvTD6mBz6b1Bw99Fa,IfcBeam,Concrete-Rectangular Beam:STI_Balk 1600x700:16...,None,None,None,1679519,Beams,"Concrete, Cast-in-Place gray",12.4,NaN,NaN,NaN,13.888,13.888,NaN,NaN,59.28,1.12,True
2,3YQplDvTD6mBz6b1Bw999y,IfcBeam,Concrete-Rectangular Beam:STI_Balk 1600x700:16...,None,None,None,1679623,Beams,"Concrete, Cast-in-Place gray",12.4,NaN,NaN,NaN,13.888,13.888,NaN,NaN,59.28,1.12,True
3,3YQplDvTD6mBz6b1Bw999n,IfcBeam,Concrete-Rectangular Beam:STI_Balk 1600x700:16...,None,None,None,1679626,Beams,"Concrete, Cast-in-Place gray",12.4,NaN,NaN,NaN,13.888,13.888,NaN,NaN,59.28,1.12,True
4,3YQplDvTD6mBz6b1Bw999s,IfcBeam,Concrete-Rectangular Beam:STI_Balk 1600x700:16...,None,None,None,1679629,Beams,"Concrete, Cast-in-Place gray",12.4,NaN,NaN,NaN,13.888,13.888,NaN,NaN,59.28,1.12,True
5,3YQplDvTD6mBz6b1Bw999h,IfcBeam,Concrete-Rectangular Beam:STI_Balk 1600x700:16...,None,None,None,1679632,Beams,"Concrete, Cast-in-Place gray",12.4,NaN,NaN,NaN,13.888,13.888,NaN,NaN,59.28,1.12,True
7,2MkfNJG9b9mQhC0P8C4Dpd,IfcColumn,M_Concrete-Rectangular-Column:STI_Betong 1200x...,None,None,None,1677653,Columns,"Concrete, Cast-in-Place gray",5.2,NaN,NaN,NaN,7.488,7.488,NaN,NaN,NaN,1.44,True
8,2MkfNJG9b9mQhC0P8C4Dpa,IfcColumn,M_Concrete-Rectangular-Column:STI_Betong 1200x...,None,None,None,1677654,Columns,"Concrete, Cast-in-Place gray",5.2,NaN,NaN,NaN,7.488,7.488,NaN,NaN,NaN,1.44,True
9,2MkfNJG9b9mQhC0P8C4Dpg,IfcColumn,M_Concrete-Rectangular-Column:STI_Betong 1200x...,None,None,None,1677656,Columns,"Concrete, Cast-in-Place gray",5.2,NaN,NaN,NaN,7.488,7.488,NaN,NaN,NaN,1.44,True
10,2MkfNJG9b9mQhC0P8C4Dph,IfcColumn,M_Concrete-Rectangular-Column:STI_Betong 1200x...,None,None,None,1677657,Columns,"Concrete, Cast-in-Place gray",5.2,NaN,NaN,NaN,7.488,7.488,NaN,NaN,NaN,1.44,True


Dataset shape: (34, 20)
Structural elements: 33


In [16]:
df_qto = (
    df_bim_structural
    .groupby("category")
    .agg(
        element_count=("global_id", "count"),
        total_volume_m3=(
            "net_volume_m3",
            lambda values: values.sum(min_count=1)
        ),
        missing_volumes=("net_volume_m3",
            lambda values: values.isna().sum())
    )
    .reset_index()
    .sort_values("total_volume_m3", ascending=False)
)

display(df_qto.round(3))

print(
    "Total concrete volume:",
    round(df_bim_structural["net_volume_m3"].sum(min_count=1), 3),
    "m³"
)

,category,element_count,total_volume_m3,missing_volumes
1,Bridge Deck,1,3645.000,0
4,Walls,2,442.260,0
3,Foundations,12,288.000,0
2,Columns,12,89.856,0
0,Beams,6,83.328,0


Total concrete volume: 4548.444 m³


In [17]:
import numpy as np

quality_summary = {
    "total_elements": len(df_bim),
    "structural_elements": len(df_bim_structural),
    "excluded_elements": len(df_bim) - len(df_bim_structural),
    "duplicate_global_ids": int(
        df_bim["global_id"].duplicated().sum()
    ),
    "missing_volumes": int(
        df_bim_structural["net_volume_m3"].isna().sum()
    ),
    "non_finite_volumes": int(
        (~np.isfinite(
            df_bim_structural["net_volume_m3"]
        )).sum()
    ),
    "non_positive_volumes": int(
        (df_bim_structural["net_volume_m3"] <= 0).sum()
    ),
    "missing_materials": int(
        df_bim_structural["material"].isna().sum()
    ),
}

display(
    pd.DataFrame(
        quality_summary.items(),
        columns=["metric", "value"]
    )
)

,metric,value
0,total_elements,34
1,structural_elements,33
2,excluded_elements,1
3,duplicate_global_ids,0
4,missing_volumes,0
5,non_finite_volumes,0
6,non_positive_volumes,0
7,missing_materials,0


In [18]:
OUTPUT_DIR = ROOT / "data/processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_bim.to_csv(
    OUTPUT_DIR / "ifc_elements.csv",
    index=False
)

df_bim_structural.to_csv(
    OUTPUT_DIR / "ifc_structural_elements.csv",
    index=False
)

df_qto.to_csv(
    OUTPUT_DIR / "ifc_quantity_summary.csv",
    index=False
)

pd.DataFrame([quality_summary]).to_csv(
    OUTPUT_DIR / "ifc_quality_summary.csv",
    index=False
)

print("All datasets exported successfully!")

All datasets exported successfully!
